# Lezione 7 — Computazione su GPU

Corso **"Deep Learning con Python e PyTorch da zero"** — DIH DATAlife, basato sulla certificazione EITC/AI/DLPP (European IT Certificate / Artificial Intelligence / Deep Learning with Python and PyTorch).

**Durata stimata della lezione: 60 minuti**

## Obiettivi di apprendimento di questa lezione

- Comprendere la differenza tra CPU e GPU e quando usare l'una o l'altra
- Trasferire modelli e dati su GPU in PyTorch
- Misurare l'accelerazione effettiva e come limitazioni di memoria (VRAM) influiscono
- Risoluzione dei problemi comuni con GPU (memoria esaurita, incompatibilità CUDA)

Questo notebook accompagna il capitolo 07 del manuale del corso. Esegui ogni cella in ordine con **Maiusc + Invio**. Per ottenere il massimo da questa lezione, eseguila in Google Colab con GPU abilitata: Modifica → Impostazioni → Acceleratore hardware → GPU.

## Parte 1 — CPU vs. GPU: perché la GPU è veloce

Una **CPU** (unità di elaborazione centrale) ha alcuni core (4-64, tipicamente) molto veloci e versatili, adatti a compiti generali.

Una **GPU** (unità di elaborazione grafica) ha migliaia di core deboli, progettati per operazioni parallele: moltiplicazioni di matrice eseguite contemporaneamente.

Il deep learning è una cascata di moltiplicazioni di matrice: il modello (matrice di pesi) moltiplicato per i dati (matrice di input), ancora e ancora. Una GPU è **praticamente nata** per questo.

Ordini di grandezza di accelerazione:

- Modelli piccoli (< 1M parametri): CPU vs. GPU, differenza minore
- Modelli medi (10M parametri): GPU 10× veloce
- Modelli grandi (100M+ parametri): GPU 50-100× veloce

Ma una GPU ha una **memoria limitata** (VRAM): mentre una CPU moderna ha 16GB+ di RAM, una GPU spesso ha 2-8GB di VRAM. Se il modello o il batch non entra, ottieni un errore "out of memory" (CUDA out of memory).

In [ ]:
import torch
import time

print("Versión de PyTorch:", torch.__version__)
print("¿Hay una GPU con CUDA disponible?", torch.cuda.is_available())

## Parte 2 — Verifica disponibilità GPU e CUDA

`torch.cuda` è il modulo di PyTorch per computazione GPU. Verifica se il tuo sistema supporta CUDA e una GPU è disponibile.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

if torch.cuda.is_available():
    print("Nombre de la GPU:", torch.cuda.get_device_name(0))
    print("Número de GPU detectadas:", torch.cuda.device_count())
else:
    print("No hay GPU activada en este entorno de ejecución: todo el cálculo correrá en CPU")

## Parte 3 — Trasferire il modello e i dati su GPU

Quando hai confermato che una GPU è disponibile, trasferisci il modello e i dati su di essa. In PyTorch, tutto ciò che ha parametri (modelli) o è un tensore (dati) può essere trasferito con `.to(device)`.

In [ ]:
tensor_cpu = torch.randn(4, 4)
print("Tensor recién creado, device:", tensor_cpu.device)

tensor_en_device = tensor_cpu.to(device)
print("Tensor movido, device:", tensor_en_device.device)

# Todos los tensores de una misma operación deben estar en el mismo device
resultado = tensor_en_device + tensor_en_device
print("Operación correcta, device del resultado:", resultado.device)

# Si descomentas la siguiente línea con una GPU activada, PyTorch lanza un
# RuntimeError porque tensor_cpu sigue en la CPU y tensor_en_device ya no:
# resultado_erroneo = tensor_cpu + tensor_en_device

## Parte 4 — Addestrare su GPU

Addestrarai la CNN su GPU. L'unica differenza dal codice su CPU è che devi trasferire il modello e i dati su GPU prima del ciclo di addestramento.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class RedMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = nn.Linear(28 * 28, 128)
        self.capa2 = nn.Linear(128, 64)
        self.capa3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)      # aplana cada imagen 28x28 en un vector de 784 valores
        x = F.relu(self.capa1(x))
        x = F.relu(self.capa2(x))
        return self.capa3(x)           # logits en bruto: CrossEntropyLoss se encarga del resto


modelo = RedMNIST().to(device)
print("Device de los parámetros del modelo:", next(modelo.parameters()).device)

## Parte 5 — Misurare l'accelerazione

Addestrarai il modello sia su CPU che su GPU per misurare il tempo di parete (wall-clock time) e calcolare lo speedup. Su una GPU debole potrebbe essere solo 2-5×, su una GPU potente (come una NVIDIA A100) potrebbe essere 50×+.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transformaciones = transforms.Compose([transforms.ToTensor()])
dataset_entrenamiento = datasets.MNIST(
    root="./datos", train=True, download=True, transform=transformaciones
)
cargador_entrenamiento = DataLoader(dataset_entrenamiento, batch_size=128, shuffle=True)

print("Número de imágenes de entrenamiento:", len(dataset_entrenamiento))

## Parte 6 — Problemi comuni e soluzioni

### "CUDA out of memory"

Il batch è troppo grande per stare in VRAM. Soluzioni:

- Riduci `batch_size`
- Usa una GPU con più memoria
- Usa Gradient Checkpointing (salva selettivamente solo alcuni gradienti intermedi per conservare memoria)

### GPU disponibile ma il modello gira lentamente

Probabilmente il modello è piccolo e il trasferimento su GPU + overhead sono più lenti del calcolo stesso. Per modelli molto piccoli, la CPU potrebbe essere più veloce.

### CUDA versione incompatibile

PyTorch è compilato per una specifica versione di CUDA (es. CUDA 11.8). Se la tua GPU usa una versione diversa, potrebbero esserci problemi di compatibilità. Controlla con `torch.version.cuda` e reinstalla PyTorch della versione giusta se necessario.

In [ ]:
import torch.optim as optim

funcion_perdida = nn.CrossEntropyLoss()
optimizador = optim.Adam(modelo.parameters(), lr=0.001)

inicio = time.time()
for imagenes, etiquetas in cargador_entrenamiento:
    imagenes = imagenes.to(device)     # el lote de imágenes viaja al device
    etiquetas = etiquetas.to(device)   # las etiquetas también, o CrossEntropyLoss fallará

    optimizador.zero_grad()
    salida = modelo(imagenes)
    perdida = funcion_perdida(salida, etiquetas)
    perdida.backward()
    optimizador.step()

tiempo_epoca = time.time() - inicio
print(f"Tiempo de una época en {device}: {tiempo_epoca:.2f} segundos")

## Riepilogo della lezione

- Una GPU ha migliaia di core paralleli, ideale per il deep learning (moltiplicazioni di matrice)
- Verifica disponibilità GPU con `torch.cuda.is_available()`
- Trasferisci modello e dati su GPU con `.to(device)` o `.cuda()`
- Aspettati accelerazione 5-50× per modelli medi, a seconda della GPU
- Se il batch non entra in VRAM, riduci `batch_size`
- Non tutte le applicazioni beneficiano della GPU: modelli molto piccoli potrebbero essere più veloci su CPU

Nel notebook di pratica addestrerai modelli di diverse dimensioni su CPU e GPU per misurare il punto di pareggio (quando la GPU diventa vantaggiosa).

## Riepilogo

- Una CPU ha pochi core molto versatili; una GPU ha migliaia di core semplici pensati per ripetere la stessa operazione su molti dati in parallelo, esattamente ciò di cui ha bisogno l'addestramento di una rete neurale
- CUDA è la piattaforma di NVIDIA che permette a PyTorch di comunicare con la GPU; non scrivi codice CUDA tu stesso
- `torch.cuda.is_available()` ti dice se è disponibile una GPU; `torch.device("cuda" if torch.cuda.is_available() else "cpu")` costruisce un `device` che funziona con o senza GPU
- `.to(device)` sposta tensori e modelli sul dispositivo scelto; tutti i tensori di una stessa operazione devono trovarsi sullo stesso `device`
- Sposti il modello sul `device` una sola volta, fuori dal ciclo di addestramento, e ogni lotto di dati dentro il ciclo, subito prima di usarlo
- La differenza di tempo tra CPU e GPU si nota soprattutto con lotti grandi e modelli con più parametri; nel notebook di pratica di questa lezione misurerai anche il costo di spostare i dati sul device